# Hybrid RAG — The Best of Both Worlds

Classic RAG keeps exact text but misses connections.  
Graph RAG finds connections but loses exact details.  
**Hybrid RAG combines both.**

```
User Question
     │
     ├──> Graph RAG: find relevant ENTITIES and their CONNECTIONS
     │    (tells us WHERE to look)
     │
     ├──> Use those entities to retrieve ORIGINAL TEXT CHUNKS
     │    (gives us the exact details, numbers, quotes)
     │
     └──> LLM answers with BOTH: structured relationships + exact text
```

**Prerequisites:** Run `kg_vs_rag_showdown.ipynb` first to populate Neo4j.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
import chromadb

load_dotenv(Path(".").resolve().parent / ".env")
load_dotenv(Path(".").resolve() / ".env")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
emb_model = OpenAIEmbeddings(model="text-embedding-3-small")

NEO4J_URI = "bolt://localhost:7687"
NEO4J_AUTH = ("neo4j", "workshop2024")

def ask_llm(prompt, system="You are a helpful assistant."):
    return llm.invoke([SystemMessage(content=system), HumanMessage(content=prompt)]).content

print("Setup complete!")

---
## Load the document + build ChromaDB index

In [ ]:
document = Path("../data/meridian_dossier.txt").read_text()
print(f"Document: {len(document)} chars, {len(document.split())} words")

# Build ChromaDB vector index (same as classic RAG)
CHUNK_SIZE = 300
chunks = [document[i:i+CHUNK_SIZE] for i in range(0, len(document), CHUNK_SIZE - 30)]
chunk_embs = emb_model.embed_documents(chunks)

chroma = chromadb.Client()
try: chroma.delete_collection("hybrid-rag")
except: pass
collection = chroma.create_collection("hybrid-rag")
collection.add(
    ids=[f"c{i}" for i in range(len(chunks))],
    embeddings=chunk_embs,
    documents=chunks,
)
print(f"ChromaDB: {collection.count()} chunks")

# Verify Neo4j has the KG (from kg_vs_rag_showdown notebook)
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
with driver.session() as s:
    nodes = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    edges = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
driver.close()
print(f"Neo4j: {nodes} nodes, {edges} edges")

if nodes < 10:
    print("\nWARNING: Neo4j looks empty. Run kg_vs_rag_showdown.ipynb first!")

---
## The Three Approaches

Let's define all three — Classic RAG, Graph RAG, and Hybrid RAG — so we can compare.

In [ ]:
def classic_rag(question, top_k=2):
    """Vector search → retrieve chunks → LLM answers."""
    q_emb = emb_model.embed_query(question)
    results = collection.query(query_embeddings=[q_emb], n_results=top_k)
    context = "\n---\n".join(results["documents"][0])
    answer = ask_llm(
        f"Context:\n{context}\n\nQuestion: {question}",
        system="Answer ONLY from context. If info is missing, say what's missing."
    )
    return answer


def graph_rag(question):
    """Graph traversal → get relationship triples → LLM answers."""
    driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
    with driver.session() as s:
        rows = [dict(r) for r in s.run(
            "MATCH (a:Entity)-[r]->(b:Entity) RETURN a.name AS src, r.type AS rel, b.name AS tgt"
        )]
    driver.close()
    context = "\n".join([f"{r['src']} --[{r['rel']}]--> {r['tgt']}" for r in rows])
    answer = ask_llm(
        f"Knowledge Graph:\n{context}\n\nQuestion: {question}",
        system="Answer using ONLY the knowledge graph relationships."
    )
    return answer


def hybrid_rag(question):
    """
    HYBRID: Graph finds the entities → vector search retrieves their original text → LLM answers.
    
    Step 1: Ask the graph "what entities are relevant?"
    Step 2: For each relevant entity, search ChromaDB for chunks mentioning it
    Step 3: Combine graph relationships + original text chunks
    Step 4: LLM answers with both structured + textual context
    """
    driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
    
    # ── Step 1: Find relevant entities in the graph ───────────────
    # Ask LLM to extract entity names from the question
    entity_str = ask_llm(
        f"List the key entity names (people, companies, products) from this question, one per line:\n{question}",
        system="Return ONLY entity names, one per line. No explanations."
    )
    search_terms = [e.strip() for e in entity_str.strip().split("\n") if e.strip()]
    print(f"  Step 1 — Entities from question: {search_terms}")
    
    # ── Step 2: Get graph relationships for those entities ────────
    graph_triples = []
    related_entity_names = set()
    with driver.session() as s:
        for term in search_terms:
            for rec in s.run(
                """MATCH (a:Entity)-[r]->(b:Entity)
                   WHERE toLower(a.name) CONTAINS toLower($t)
                      OR toLower(b.name) CONTAINS toLower($t)
                   RETURN a.name AS src, r.type AS rel, b.name AS tgt
                   LIMIT 20""", t=term):
                graph_triples.append(f"{rec['src']} --[{rec['rel']}]--> {rec['tgt']}")
                related_entity_names.add(rec['src'])
                related_entity_names.add(rec['tgt'])
    driver.close()
    
    graph_context = "\n".join(list(set(graph_triples)))
    print(f"  Step 2 — Graph triples found: {len(set(graph_triples))}")
    print(f"  Step 2 — Related entities discovered: {related_entity_names}")
    
    # ── Step 3: Retrieve original text chunks for those entities ──
    # Search ChromaDB for each entity name found in the graph
    all_search_terms = list(search_terms) + list(related_entity_names)
    text_chunks = set()
    for entity_name in all_search_terms[:8]:  # limit to avoid too much context
        q_emb = emb_model.embed_query(entity_name)
        results = collection.query(query_embeddings=[q_emb], n_results=1)
        for doc in results["documents"][0]:
            text_chunks.add(doc)
    
    text_context = "\n---\n".join(list(text_chunks))
    print(f"  Step 3 — Text chunks retrieved: {len(text_chunks)}")
    
    # ── Step 4: LLM answers with BOTH graph + text ────────────────
    combined_context = f"""GRAPH RELATIONSHIPS (structured connections):
{graph_context}

ORIGINAL TEXT (exact details, numbers, dates):
{text_context}"""
    
    answer = ask_llm(
        f"Combined context:\n{combined_context}\n\nQuestion: {question}",
        system="""Answer using BOTH the graph relationships AND the original text.
Use the graph to understand connections between entities.
Use the original text for exact numbers, dates, and details.
Be comprehensive and specific."""
    )
    return answer

print("All three approaches ready!")

---
## The Comparison

Same questions, three approaches side by side.

In [ ]:
def compare_all(question, q_num):
    print(f"\n{'━'*70}")
    print(f"  Q{q_num}: {question}")
    print(f"{'━'*70}")
    
    print("\n[1/3] Running Classic RAG...")
    r = classic_rag(question)
    
    print("[2/3] Running Graph RAG...")
    g = graph_rag(question)
    
    print("[3/3] Running Hybrid RAG...")
    h = hybrid_rag(question)
    
    print(f"\n┌─ CLASSIC RAG {'─'*54}┐")
    for line in r.split(". ")[:3]:
        print(f"│ {line.strip()[:66]}")
    print(f"└{'─'*68}┘")
    
    print(f"\n┌─ GRAPH RAG {'─'*56}┐")
    for line in g.split(". ")[:3]:
        print(f"│ {line.strip()[:66]}")
    print(f"└{'─'*68}┘")
    
    print(f"\n┌─ HYBRID RAG {'─'*55}┐")
    for line in h.split(". ")[:5]:
        print(f"│ {line.strip()[:66]}")
    print(f"└{'─'*68}┘")

print("compare_all() ready!")

### Q1: Simple Fact (RAG should win, Hybrid matches)

In [ ]:
compare_all("When was Meridian founded and by whom?", 1)

### Q2: Multi-Hop — Who left where? (KG wins, Hybrid dominates)

In [ ]:
compare_all("List every person who left Meridian, where they went, and their new role.", 2)

### Q3: Cross-Section — All funding rounds (RAG partial, KG fails, Hybrid wins)

In [ ]:
compare_all("List ALL three funding rounds with exact amounts, lead investors, and all participants.", 3)

### Q4: Relationship Chain (RAG fails, KG partial, Hybrid best)

In [ ]:
compare_all("Trace the COMPLETE history between Amara Osei and Robert Zhang — from J&J to reconciliation.", 4)

### Q5: 4-Hop Path (RAG impossible, KG finds path, Hybrid adds detail)

In [ ]:
compare_all("What path connects angel investor Thomas Lee to the Illumina acquisition? Trace every step.", 5)

### Q6: Details + Connections (Hybrid's sweet spot)

In [ ]:
compare_all("What was the BioNexus lawsuit about? Who were the lawyers, what was the patent number, and how was it settled?", 6)

---
## Summary — When to Use What

| Question Type | Classic RAG | Graph RAG | Hybrid RAG |
|--------------|-------------|-----------|------------|
| Simple fact (date, name) | **Good** | Overkill | Good |
| Exact numbers ($120M, 78% → 94%) | **Best** | Loses numbers | **Best** |
| Who left where (aggregation) | Gets 1-2 | **Gets 4-5** | **Gets all + details** |
| Multi-hop path (A→B→C→D) | **Fails** | **Finds path** | **Path + context** |
| Cross-section (all funding rounds) | Gets 1 of 3 | Misses amounts | **Gets all with amounts** |
| Relationship history | Partial | Structural | **Complete story** |

### The Verdict

**Hybrid RAG is the production answer.** It uses the knowledge graph to know WHERE to look, then retrieves the original text for the actual details. Neither approach alone is sufficient:

- **Classic RAG alone**: misses connections between distant parts of the document
- **Graph RAG alone**: loses exact numbers, dates, and nuanced details during extraction
- **Hybrid RAG**: graph provides the structure, text provides the substance

### How Hybrid RAG works
```
Question: "What path connects Thomas Lee to Illumina?"

Step 1 — Extract entities from question: [Thomas Lee, Illumina]
Step 2 — Graph traversal finds: Thomas Lee → Meridian → GenomicInsight → Illumina
Step 3 — Retrieve text chunks mentioning each entity in the path
Step 4 — LLM gets BOTH the path AND the text with exact details:
         "Thomas Lee invested $250K in the seed round..."
         "GenomicInsight was acquired for $45M..."
         "BioNexus was acquired by Illumina for $500M..."
```